# Lab 5 — Part 2: Quota, billing and pre-run estimate

ใช้ WSL และ kernel `itcs355-lab3` เดิม **ไม่ต้องรัน Part 1 ซ้ำ** — อ่านผลจาก `reports/preflight.json`

รัน code cell เดียวด้านล่าง จะ:
1. อ่าน quota ใหม่ด้วย parser ที่แก้แล้ว และอ่านรายชื่อ managed identities เดิม
2. อ่าน posted billing ของ resource group วิชาในช่วง 1 กันยายน 2026 ถึงเวลารัน และ query แยก tags `student=6688012, lab=5`
3. สร้าง `reports/lab5-estimate.md` จากราคาใน Part 1 และสมมติฐานสาธิตระยะสั้น
4. เตรียมหัวข้อ `reports/lab5-cost.md` โดยเว้นผล actual/optimisation ที่ยังไม่มีหลักฐานไว้เป็น pending

**ไม่มีการสร้าง resource, ถอน/เพิ่มสิทธิ์, training, install, commit หรือ push**

ราคาจาก Part 1 ต้องไม่เก่าเกิน 24 ชั่วโมง สมมติฐานค่าใช้จ่ายเป็นข้อเสนอสำหรับตรวจทาน:
training 0.5 ชั่วโมง + pipeline/startup/idle 0.5 ชั่วโมง + staging 1 ชั่วโมง
เงินสำรอง storage/monitoring/registry/LLM/provider ที่สองระบุแยกชัดเจนว่าไม่ใช่ราคาที่วัดหรือใบเสนอราคา
Part 2 ไม่แก้ทับ estimate/cost report ที่มีอยู่แล้ว

ถ้าขั้น billing ถูกปฏิเสธจะบันทึกว่า unavailable; ไม่รายงานว่า actual เป็นศูนย์
quota อ่านได้ไม่ได้แปลว่า capacity/สิทธิ์พอ และ billing ช่วงนี้ยังไม่ยืนยันงบทั้งเทอม
ถ้าไม่มี progress ใหม่เกิน 5 นาทีให้ Interrupt แล้วส่ง output ล่าสุด


In [1]:
from pathlib import Path
import platform
import subprocess
import sys

if platform.system() != "Linux" or "microsoft" not in platform.release().lower():
    raise RuntimeError("เลือก kernel itcs355-lab3 ฝั่ง WSL ก่อนรันครับ")
here = Path.cwd().resolve()
lab = next((base / "lab5" for base in (here, *here.parents)
            if (base / "lab5" / "scripts" / "plan_run.py").is_file()
            and (base / "lab4" / "README.md").is_file()), None)
if lab is None:
    raise RuntimeError("เปิด notebook จาก repository เดิมผ่าน WSL ก่อนครับ")
print("Part 2: quota + billing + pre-run estimate; no resource creation or Git changes.", flush=True)
with subprocess.Popen(
    [sys.executable, "-u", "-m", "scripts.plan_run"], cwd=lab,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
) as task:
    try:
        for line in task.stdout:
            print(line, end="", flush=True)
        exit_code = task.wait()
    except KeyboardInterrupt:
        task.terminate()
        try:
            task.wait(timeout=10)
        except subprocess.TimeoutExpired:
            task.kill()
        raise
if exit_code:
    raise RuntimeError("Part 2 ยังไม่ผ่าน ส่ง summary ด้านบนให้ผู้ช่วยตรวจครับ")
print("Part 2 เสร็จ: ส่ง summary ก่อนเริ่ม commit หรือ managed pipeline ครับ", flush=True)


Part 2: quota + billing + pre-run estimate; no resource creation or Git changes.
[13:21:10] 1/4 Recheck quota using the corrected SDK response parser
[13:21:13] 2/4 Read existing managed identities; no permission changes
[13:21:15] 3/4 Query posted billing for the course resource group and Lab 5 tags
[13:21:21] 4/4 Prepare the pre-run estimate and cost-report sections

--- SEND THIS SUMMARY ---
{
  "status": "ready_for_review",
  "estimate": {
    "status": "draft_created",
    "path": "lab5-estimate.md",
    "hourly_thb": 5.6784,
    "part_a_subtotal_thb": 17.3568,
    "proposed_envelope_thb": 44.8282,
    "meter_id": "8ff981a1-40b8-4292-aa18-c1af57836008",
    "sha256": "65e0795089f3748c99a08c9abd44eff94d2558939e24ff7c02f648de7247350a"
  },
  "quota": {
    "status": "ok",
    "relevant": [
      {
        "name": "TotalClusters",
        "current": 0,
        "limit": 200,
        "unit": "Count"
      },
      {
        "name": "TotalDedicatedCores",
        "current": 0,
        "

## ส่งผลแล้วหยุดตรงนี้

คัดลอก output ตั้งแต่ `SEND THIS SUMMARY` กลับมา ไม่ต้องส่ง key หรือ cloud.env
ดูร่าง estimate ได้ใน `lab5/reports/lab5-estimate.md`

ต้องตรวจ quota/สิทธิ์/งบและ commit estimate **ก่อน** ส่ง managed pipeline ตาม Task 1
notebook นี้ไม่ commit ให้อัตโนมัติ ผู้ช่วยจะเตรียมขั้นถัดไปจากผลจริง

อ้างอิง: [SDK quota response รุ่น 1.35.0](https://github.com/Azure/azure-sdk-for-python/blob/azure-ai-ml_1.35.0/sdk/ml/azure-ai-ml/azure/ai/ml/entities/_compute/_usage.py),
[Cost Management Query](https://learn.microsoft.com/en-us/rest/api/cost-management/query/usage?view=rest-cost-management-2025-03-01)
